In [0]:
%pip install gtfs-realtime-bindings
dbutils.library.restartPython()

In [0]:
from google.transit import gtfs_realtime_pb2
from pyspark.sql.functions import udf, col, explode
from pyspark.sql.functions import from_unixtime, to_timestamp, col
from pyspark.sql.types import (
    ArrayType, StructType, StructField,
    StringType, DoubleType, LongType, IntegerType)

catalog = "transit_analytics"
bronze_schema = "bronze"
silver_schema = "silver"

In [0]:
vehicle_position_schema = ArrayType(StructType([
    StructField("entity_id", StringType()),
    StructField("trip_id", StringType()),
    StructField("route_id", StringType()),
    StructField("direction_id", IntegerType()),
    StructField("start_date", StringType()),
    StructField("latitude", DoubleType()),
    StructField("longitude", DoubleType()),
    StructField("bearing", DoubleType()),
    StructField("speed", DoubleType()),
    StructField("vehicle_id", StringType()),
    StructField("vehicle_timestamp", LongType()),
]))


@udf(returnType=vehicle_position_schema)
def parse_vehicle_positions(raw_bytes: bytes) -> list[dict]:
    """Decode a GTFS-RT VehiclePositions protobuf blob into structured rows."""
    feed = gtfs_realtime_pb2.FeedMessage()
    feed.ParseFromString(raw_bytes)

    rows = []
    for entity in feed.entity:
        v = entity.vehicle
        rows.append({
            "entity_id": entity.id,
            "trip_id": v.trip.trip_id,
            "route_id": v.trip.route_id,
            "direction_id": v.trip.direction_id,
            "start_date": v.trip.start_date,
            "latitude": v.position.latitude,
            "longitude": v.position.longitude,
            "bearing": v.position.bearing,
            "speed": v.position.speed,
            "vehicle_id": v.vehicle.id,
            "vehicle_timestamp": v.timestamp,
        })
    return rows


trip_update_schema = ArrayType(StructType([
    StructField("entity_id", StringType()),
    StructField("trip_id", StringType()),
    StructField("route_id", StringType()),
    StructField("start_date", StringType()),
    StructField("vehicle_id", StringType()),
    StructField("stop_sequence", IntegerType()),
    StructField("stop_id", StringType()),
    StructField("arrival_time", LongType()),
    StructField("feed_timestamp", LongType()),
]))


@udf(returnType=trip_update_schema)
def parse_trip_updates(raw_bytes: bytes) -> list[dict]:
    """Decode a GTFS-RT TripUpdates protobuf blob into structured rows (one per stop)."""
    feed = gtfs_realtime_pb2.FeedMessage()
    feed.ParseFromString(raw_bytes)

    rows = []
    for entity in feed.entity:
        tu = entity.trip_update
        for stu in tu.stop_time_update:
            rows.append({
                "entity_id": entity.id,
                "trip_id": tu.trip.trip_id,
                "route_id": tu.trip.route_id,
                "start_date": tu.trip.start_date,
                "vehicle_id": tu.vehicle.id,
                "stop_sequence": stu.stop_sequence,
                "stop_id": stu.stop_id,
                "arrival_time": stu.arrival.time,
                "feed_timestamp": tu.timestamp,
            })
    return rows

In [0]:
from pyspark.sql.functions import date_format, from_utc_timestamp

bronze_vp_df = spark.table(f"{catalog}.{bronze_schema}.vehicle_positions")

silver_vp_df = (
    bronze_vp_df
    .withColumn("parsed", parse_vehicle_positions(col("content")))
    .withColumn("record", explode(col("parsed")))
    .select(
        "record.entity_id",
        "record.trip_id",
        "record.route_id",
        "record.direction_id",
        "record.start_date",
        "record.latitude",
        "record.longitude",
        "record.bearing",
        "record.speed",
        "record.vehicle_id",
        from_utc_timestamp(
            to_timestamp(from_unixtime(col("record.vehicle_timestamp"))), "Australia/Adelaide"
        ).alias("vehicle_ts"),
        col("path").alias("source_file"),
    )
    .filter(col("latitude").isNotNull() & col("longitude").isNotNull())
    .filter((col("latitude") != 0) & (col("longitude") != 0))
    .filter(col("speed") >= 0)
    .dropDuplicates(["entity_id", "vehicle_ts"])
    .withColumn("vehicle_date", date_format(col("vehicle_ts"), "yyyy-MM-dd"))
    .withColumn("vehicle_time", date_format(col("vehicle_ts"), "HH:mm:ss"))
    .orderBy("vehicle_ts")
)

silver_vp_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(
    f"{catalog}.{silver_schema}.vehicle_positions"
)

print("vehicle_positions row count:", spark.table(f"{catalog}.{silver_schema}.vehicle_positions").count())

In [0]:
bronze_tu_df = spark.table(f"{catalog}.{bronze_schema}.trip_updates")

silver_tu_df = (
    bronze_tu_df
    .withColumn("parsed", parse_trip_updates(col("content")))
    .withColumn("record", explode(col("parsed")))
    .select(
        "record.entity_id",
        "record.trip_id",
        "record.route_id",
        "record.start_date",
        "record.vehicle_id",
        "record.stop_sequence",
        "record.stop_id",
        from_utc_timestamp(
            to_timestamp(from_unixtime(col("record.arrival_time"))), "Australia/Adelaide"
        ).alias("arrival_ts"),
        from_utc_timestamp(
            to_timestamp(from_unixtime(col("record.feed_timestamp"))), "Australia/Adelaide"
        ).alias("feed_ts"),
        col("path").alias("source_file"),
    )
    .filter(col("arrival_ts").isNotNull())
    .dropDuplicates(["trip_id", "stop_sequence", "feed_ts"])
    .withColumn("arrival_date", date_format(col("arrival_ts"), "yyyy-MM-dd"))
    .withColumn("arrival_time_only", date_format(col("arrival_ts"), "HH:mm:ss"))
    .orderBy("trip_id", "stop_sequence")
)

silver_tu_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(
    f"{catalog}.{silver_schema}.trip_updates"
)

print("trip_updates row count:", spark.table(f"{catalog}.{silver_schema}.trip_updates").count())

In [0]:
# Schema for both tables
spark.table(f"{catalog}.{silver_schema}.vehicle_positions").printSchema()
spark.table(f"{catalog}.{silver_schema}.trip_updates").printSchema()

# 2 sample rows for both tables
display(spark.table(f"{catalog}.{silver_schema}.vehicle_positions").limit(2))
display(spark.table(f"{catalog}.{silver_schema}.trip_updates").limit(2))
print("")